In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import time
import nibabel as nib
import os
import json
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from pathlib import Path
from torch.optim import Adam
from scipy.ndimage import distance_transform_edt
import time
from typing import Dict, Optional

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [18]:
#mount data drive
from google.colab import drive

drive.mount('/content/drive') # mount drive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
print(device)

cuda


In [20]:
def z_score_normalization(x: torch.tensor, eps=1e-8) -> torch.tensor:
    """
    Args:
        - x: 4D tensor of shape (channel, depth, height, width)
        - eps: float to avoid dividing by zero
    performs per-channel z-score normalization, avoids supressing signals or enforcing artificial signals
    return:
        - output: a 4D tensor of shape (channel, depth, height, width)
                    - each channel is normalized seperately and is added to its respective position in the output tensor
                    using its own mean and standard deviation
    """

    # initializing output tensor with the same shape as input tensor
    output = torch.empty_like(x)

    for c in range(x.shape[0]):
        channel = x[c] # shape (depth, width, height)

        mean = channel.mean()
        std = channel.std() + eps

        output[c] = (channel-mean) / std

    return output

In [21]:
# method to load .nii.gz files
def load_nib(file_path, filename):
    """
    loads .nii.gz files and returns its contents as nunpy array
    Args:
        - file_path: file path or directory of the files
        - filename: name of the file to be loaded

    return:
        - numpy format of the file
    """
    return nib.load(os.path.join(file_path, filename)).get_fdata()

In [22]:

def filename_set(file_path):
    """
    Args:
        file_path: file path or directory of the files

    return:
        - list of filenames that ends with .nii.gz
    """
    return {f for f in os.listdir(file_path) if f.endswith( '.nii.gz')}

In [23]:
class BratsDataset(Dataset):
    """
    creates pytorch dataset to load .nii.gz images and labels
    image and label inputs are of the foramt (height, width, depth, channel)
    image_output format: (channel, depth, height, width)
    label_output format: (classes, depth, height, width)
    """


    def __init__(self, image_dir, labels_dir, transform=None):
        """
        Args
            - image_dir: directory containing image files
            - label_dir: directory containing label files
            - transform: callable method that transforms the data
        """

        self.image_dir = image_dir # image directory
        self.label_dir = labels_dir # label directory

        # gets set of file names from image and label directories
        self.image_files = filename_set(self.image_dir)
        self.label_files = filename_set(self.label_dir)

        self.transform = transform # loads transform

        # stores intersection of file names that are in both image and label directories
        self.common_filenames = sorted(self.image_files & self.label_files)


    def __len__(self):
        # returns length of files that are common in image and label directories
        return len(self.common_filenames)


    def __getitem__(self, index):
        #  loads and returns a single sample file by applying the given transformation
        file_index = self.common_filenames[index] # indexed filename from common_filenames

        image = load_nib(self.image_dir, file_index) # loads images of .nii.gz format from images directory
        label = load_nib(self.label_dir, file_index) # loads labels of .nii.gz format from labels directory

        image_tensor = torch.from_numpy(image).float() # converts numpy format of images to float tensors
        image_tensor = image_tensor.permute(3, 2, 0, 1) # permutes (height, width, depth, channels) -> (channels, depth, height, width)

        image_tensor = z_score_normalization(image_tensor)

        label_tensor = torch.from_numpy(label) # converts numpy format of images to long tensors
        label_tensor = label_tensor.permute(2, 0, 1).long() # permutes (height, width, depth) -> (depth, height, width)


        return image_tensor, label_tensor # returns image and label tensor

In [24]:
class NormalizedActivation(nn.Module):
    def __init__(self, channels, group_size=4, eps=1e-4):
        super().__init__()
        assert channels % group_size == 0, f"Channels {channels} must be divisible by group_size {group_size}"
        self.channels = channels
        self.group_size = group_size
        self.num_groups = channels // group_size
        self.eps = eps

        # Parameters per group
        self.mu = nn.Parameter(torch.zeros(self.num_groups))
        self.alpha = nn.Parameter(torch.ones(self.num_groups))
        self.beta = nn.Parameter(torch.ones(self.num_groups))
        self.gamma = nn.Parameter(torch.ones(self.num_groups))

    def forward(self, x):
        B, C, D, H, W = x.shape


        x_grouped = x.view(B, self.num_groups, self.group_size, D, H, W)

        # Reshape parameters for broadcasting [1, num_groups, 1, 1, 1, 1]
        params_shape = (1, self.num_groups, 1, 1, 1, 1)
        mu = self.mu.view(params_shape)
        alpha = F.softplus(self.alpha).view(params_shape) + self.eps
        beta = self.beta.view(params_shape)
        gamma = self.gamma.view(params_shape)

        # Apply normalization
        gx = (x_grouped - mu) / alpha
        gx = torch.tanh(gx / gamma) * torch.abs(gamma)
        gx = torch.sigmoid(beta * gx)

        return gx.reshape(B, C, D, H, W)


In [25]:

def conv_norm_act(in_channels, out_channels, kernel=3, padding=1, stride=1):

    """
    Args:
        - in_channels: number of input channels
        - out_channels: number of output channels
        - kernel: size of convolutional kernel/filter
        - padding: size of padding
        - stride: stride of convolutional filter

        peroforms 3D convolution on the input data takes input_channels,
            if stride == 1: returns the same spatial deimensional tensor
            if stride == 2: returns the tensor with half the spatial dimension

        applies normalized activation: performs normalization followed by activation

    Returns:
        - tensor of shape (1, out_channels, depth, width, height)

        ex: if stride==2, in=16, out=32
             input: (1, 16, 78, 120, 120) --conv3d-->(1, 32, 39, 60, 60)

    """

    return nn.Sequential(
        nn.Conv3d(in_channels, out_channels, kernel_size=kernel, padding=padding, stride=stride),
        NormalizedActivation(out_channels)

    )

def conv_norm_act_upsample(in_channels, out_channels, target_size, kernel=3, padding=1, stride=1):
    """
    Args:
        - in_channels: number of input channels
        - out_channels: number of output channels
        - target_size: size of the upsampled tensor
        - kernel: size of convolutional kernel/filter
        - padding: size of padding
        - stride: stride of convolutional filter


        applies upsample and reshapes the tensor to the size of target_size

        peroforms 3D convolution on the input data takes input_channels,
            if stride == 1: returns the same spatial deimensional tensor
            if stride == 2: returns the tensor wi2,th half the spatial dimension

        applies normalized activation: performs normalization followed by activation


    Returns:
        - tensor of shape (1, out_channels, depth, width, height)


        ex: input: (1, 128, 19, 30, 30), target_shape: (1, 64, 39, 60, 60)
            upsample: (1, 128, 19, 30, 30) -> (1, 128, 39, 60, 60) -- conv --> (1, 64, 39, 60, 60)

    """
    return nn.Sequential(
        nn.Upsample(size=target_size, mode='trilinear', align_corners=True),
        conv_norm_act(in_channels, out_channels, kernel=3)
    )





In [26]:
class UNet3D(nn.Module):
    def __init__(self, in_channels=4, num_classes=4, base_channels=8):
        super().__init__()
        chs = [base_channels * (2**i) for i in range(5)]  # [8, 16, 32, 64, 128]

        # Encoder (Downsampling path)
        self.encoder = nn.ModuleList([
            conv_norm_act(in_channels, chs[0]),              # (B, 4, D, H, W) → (B, 8, D, H, W)
            conv_norm_act(chs[0], chs[1], stride=2),          # (B, 8, ...) → (B, 16, D/2, H/2, W/2)
            conv_norm_act(chs[1], chs[2], stride=2),          # ... → (B, 32, D/4, H/4, W/4)
            conv_norm_act(chs[2], chs[3], stride=2),          # ... → (B, 64, D/8, H/8, W/8)
            conv_norm_act(chs[3], chs[4], stride=2)           # ... → (B, 128, D/16, H/16, W/16)
        ])

        # Bottleneck (Bottom layer)
        self.bottleneck = conv_norm_act(chs[4], chs[4] * 2)   # (B, 128, ...) → (B, 256, ...)

        # Decoder (Upsampling path with proper skip connections)
        self.decoder = nn.ModuleList([
            nn.Sequential(
                conv_norm_act_upsample(256, 128, target_size=(10, 15, 15)),  # (B, 256, ...) → (B, 128, ...)
                conv_norm_act(128 + 128, 128)                           # After cat: (B, 256, ...) → (B, 128, ...)
            ),
            nn.Sequential(
                conv_norm_act_upsample(128, 64, target_size=(20, 30, 30)),  # (B, 128, ...) → (B, 64, ...)
                conv_norm_act(64 + 64, 64)                              # After cat: (B, 128, ...) → (B, 64, ...)
            ),
            nn.Sequential(
                conv_norm_act_upsample(64, 32, target_size=(39, 60, 60)),   # (B, 64, ...) → (B, 32, ...)
                conv_norm_act(32 + 32, 32)                             # After cat: (B, 64, ...) → (B, 32, ...)
            ),
            nn.Sequential(
                conv_norm_act_upsample(32, 16, target_size=(78, 120, 120)), # (B, 32, ...) → (B, 16, ...)
                conv_norm_act(16 + 16, 16)                             # After cat: (B, 32, ...) → (B, 16, ...)
            ),
            nn.Sequential(
                conv_norm_act_upsample(16, 8, target_size=(155, 240, 240)), # (B, 16, ...) → (B, 8, ...)
                conv_norm_act(8 + 8, 8)                                # After cat: (B, 16, ...) → (B, 8, ...)
            )
        ])

        # Final output with softmax for Dice
        self.final_conv = nn.Sequential(
            conv_norm_act(8, num_classes, kernel=1, padding=0),  # (B, 8, ...) → (B, 4, ...)
            nn.Softmax(dim=1)  # Critical for Dice loss!
        )

    def forward(self, x):
        skips = []

        # Encoder
        for layer in self.encoder:
            x = layer(x)
            skips.append(x)

        # Bottleneck
        x = self.bottleneck(x)

        # Decoder (reverse skips)
        for i, (layer, skip) in enumerate(zip(self.decoder, reversed(skips))):
            x = layer[0](x)          # Upsample + conv
            x = torch.cat([x, skip], dim=1)  # Concatenate with skip
            x = layer[1](x)          # Merge features

        return self.final_conv(x)  # Softmax-applied output

In [27]:
model = UNet3D()
x = torch.randn(size=(1, 4, 155, 240, 240), dtype=torch.float32)

In [28]:
"""
"tensorImageSize": "4D",
"modality": {
	 "0": "FLAIR",
	 "1": "T1w",
	 "2": "t1gd",
	 "3": "T2w"
 },
 "labels": {
	 "0": "background",
	 "1": "edema",
	 "2": "non-enhancing tumor",
	 "3": "enhancing tumour"
 },
"""

braintumor_datadir = "/content/drive/MyDrive/data/med/braintumor/train" # load braintumor data directory
braintumor_dataimages = os.path.join(braintumor_datadir, "images") # load brain tumor imagees directory
braintumor_datalabels = os.path.join(braintumor_datadir, "labels") # load brain tumor labels directory



In [29]:
def inspect_dataset(dataset, num_samples=10):
    print(f"legth: {len(dataset)}")
    for i in range(num_samples):

        image, label = dataset[i]
        print(f'file: {dataset.common_filenames[i]}, image_shape: {image.shape}, label_shape: {label.shape}')

In [30]:
bt_dataset = BratsDataset(braintumor_dataimages, braintumor_datalabels)

In [31]:
bt_dataloader = DataLoader(bt_dataset, batch_size=1, drop_last=True)

In [32]:
# Multi-class 3D segmentation loss functions (PyTorch)

import torch
import torch.nn as nn
import torch.nn.functional as F


def boundary_map_3d(mask):
    device = mask.device
    dtype = mask.dtype

    sobel_x = torch.tensor([
        [[-1, 0, 1], [-3, 0, 3], [-1, 0, 1]],
        [[-3, 0, 3], [-6, 0, 6], [-3, 0, 3]],
        [[-1, 0, 1], [-3, 0, 3], [-1, 0, 1]]
    ], device=device, dtype=dtype).unsqueeze(0).unsqueeze(0) / 32.0

    sobel_y = sobel_x.permute(0, 1, 3, 2, 4)
    sobel_z = sobel_x.permute(0, 1, 4, 3, 2)

    gx = F.conv3d(mask, sobel_x, padding=1)
    gy = F.conv3d(mask, sobel_y, padding=1)
    gz = F.conv3d(mask, sobel_z, padding=1)

    return torch.sqrt(gx**2 + gy**2 + gz**2)


def tversky_loss_mc(pred, target, alpha=0.3, beta=0.7, smooth=1e-6):
    num_classes = pred.shape[1]
    target_onehot = F.one_hot(target, num_classes).permute(0, 4, 1, 2, 3).float()

    pred = pred.contiguous().view(pred.size(0), pred.size(1), -1)
    target_onehot = target_onehot.contiguous().view(target_onehot.size(0), target_onehot.size(1), -1)

    TP = (pred * target_onehot).sum(dim=2)
    FP = (pred * (1 - target_onehot)).sum(dim=2)
    FN = ((1 - pred) * target_onehot).sum(dim=2)

    tversky = (TP + smooth) / (TP + alpha * FP + beta * FN + smooth)
    return (1.0 - tversky).mean()


def focal_tversky_loss_mc(pred, target, alpha=0.3, beta=0.7, gamma=1.33, smooth=1e-6):
    num_classes = pred.shape[1]
    target_onehot = F.one_hot(target, num_classes).permute(0, 4, 1, 2, 3).float()

    pred = pred.contiguous().view(pred.size(0), pred.size(1), -1)
    target_onehot = target_onehot.contiguous().view(target_onehot.size(0), target_onehot.size(1), -1)

    TP = (pred * target_onehot).sum(dim=2)
    FP = (pred * (1 - target_onehot)).sum(dim=2)
    FN = ((1 - pred) * target_onehot).sum(dim=2)

    tversky = (TP + smooth) / (TP + alpha * FP + beta * FN + smooth)
    return torch.pow(1.0 - tversky, gamma).mean()


def boundary_weighted_ce_mc(pred, target, boundary_weight=5.0):
    num_classes = pred.shape[1]
    target_onehot = F.one_hot(target, num_classes).permute(0, 4, 1, 2, 3).float()

    ce = F.cross_entropy(pred, target, reduction='none')  # (N, D, H, W)
    b_weights = torch.zeros_like(ce)

    for c in range(num_classes):
        class_map = target_onehot[:, c:c+1]  # (N,1,D,H,W)
        boundary = boundary_map_3d(class_map)
        b_weights += boundary_weight * boundary.squeeze(1)

    b_weights = 1 + b_weights
    return (b_weights * ce).mean()


class CombinedSegLoss3D_MC(nn.Module):
    def __init__(self,
                 alpha=0.3, beta=0.7, gamma=1.33,
                 lambda_tversky=1.0,
                 lambda_focal=0.5,
                 lambda_bce=0.5,
                 boundary_weight=5.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.lambda_tversky = lambda_tversky
        self.lambda_focal = lambda_focal
        self.lambda_bce = lambda_bce
        self.boundary_weight = boundary_weight

    def forward(self, pred, target):
        pred = torch.clamp(pred, 1e-4, 1 - 1e-4)
        pred_soft = F.softmax(pred, dim=1)

        loss_tversky = tversky_loss_mc(pred_soft, target, self.alpha, self.beta)
        loss_focal = focal_tversky_loss_mc(pred_soft, target, self.alpha, self.beta, self.gamma)
        loss_bce = boundary_weighted_ce_mc(pred_soft, target, self.boundary_weight)

        total = (
            self.lambda_tversky * loss_tversky +
            self.lambda_focal * loss_focal +
            self.lambda_bce * loss_bce
        )
        return total, {
            'tversky': loss_tversky.item(),
            'focal': loss_focal.item(),
            'boundary_ce': loss_bce.item()
        }


In [33]:
def tversky_loss_mc(pred, target, alpha=0.3, beta=0.7, smooth=1e-6):
    # pred: (N, C, D, H, W), target: (N, D, H, W)
    num_classes = pred.shape[1]
    target_onehot = F.one_hot(target, num_classes).permute(0, 4, 1, 2, 3).float()

    pred = pred.contiguous().view(pred.size(0), pred.size(1), -1)
    target_onehot = target_onehot.contiguous().view(target_onehot.size(0), target_onehot.size(1), -1)

    TP = (pred * target_onehot).sum(dim=2)
    FP = (pred * (1 - target_onehot)).sum(dim=2)
    FN = ((1 - pred) * target_onehot).sum(dim=2)

    tversky = (TP + smooth) / (TP + alpha * FP + beta * FN + smooth)
    return (1.0 - tversky).mean()


def focal_tversky_loss_mc(pred, target, alpha=0.3, beta=0.7, gamma=1.33, smooth=1e-6):
    num_classes = pred.shape[1]
    target_onehot = F.one_hot(target, num_classes).permute(0, 4, 1, 2, 3).float()

    pred = pred.contiguous().view(pred.size(0), pred.size(1), -1)
    target_onehot = target_onehot.contiguous().view(target_onehot.size(0), target_onehot.size(1), -1)

    TP = (pred * target_onehot).sum(dim=2)
    FP = (pred * (1 - target_onehot)).sum(dim=2)
    FN = ((1 - pred) * target_onehot).sum(dim=2)

    tversky = (TP + smooth) / (TP + alpha * FP + beta * FN + smooth)
    return torch.pow(1.0 - tversky, gamma).mean()

def boundary_weighted_ce_mc(pred, target, boundary_weight=5.0):
    # pred: (N, C, D, H, W), target: (N, D, H, W)
    num_classes = pred.shape[1]
    target_onehot = F.one_hot(target, num_classes).permute(0, 4, 1, 2, 3).float()

    ce = F.cross_entropy(pred, target, reduction='none')  # (N, D, H, W)

    b_weights = torch.zeros_like(ce)
    for c in range(num_classes):
        class_map = target_onehot[:, c:c+1]  # shape (N,1,D,H,W)
        boundary = boundary_map_3d(class_map)
        b_weights += boundary_weight * boundary.squeeze(1)

    b_weights = 1 + b_weights
    return (b_weights * ce).mean()

class CombinedSegLoss3D_MC(nn.Module):
    def __init__(self,
                 alpha=0.3, beta=0.7, gamma=1.33,
                 lambda_tversky=1.0,
                 lambda_focal=0.5,
                 lambda_bce=0.5,
                 boundary_weight=5.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.lambda_tversky = lambda_tversky
        self.lambda_focal = lambda_focal
        self.lambda_bce = lambda_bce
        self.boundary_weight = boundary_weight

    def forward(self, pred, target):
        pred = torch.clamp(pred, 1e-4, 1 - 1e-4)
        pred_soft = F.softmax(pred, dim=1)

        loss_tversky = tversky_loss_mc(pred_soft, target, self.alpha, self.beta)
        loss_focal = focal_tversky_loss_mc(pred_soft, target, self.alpha, self.beta, self.gamma)
        loss_bce = boundary_weighted_ce_mc(pred_soft, target, self.boundary_weight)

        total = (
            self.lambda_tversky * loss_tversky +
            self.lambda_focal * loss_focal +
            self.lambda_bce * loss_bce
        )
        return total, {
            'tversky': loss_tversky.item(),
            'focal': loss_focal.item(),
            'boundary_ce': loss_bce.item()
        }




In [34]:
class TverskyDiceLoss(nn.Module):

    def __init__(self, alpha=0.3, beta=0.7, eps=1e-6, reg_weight=0.1):
        """
        Args:
            - alpha: weight for false positive
            - beta: weight for false negative
            - eps: small value to avoid division by zero
            - reg_weight: egularization weight, prevents alpha beta from diverging too much
        """
        super().__init__()
        self.log_alpha = nn.Parameter(torch.log(torch.tensor(alpha)))
        self.log_beta = nn.Parameter(torch.log(torch.tensor(beta)))
        self.eps = eps
        self.reg_weight = reg_weight

    def forward(self, pred, target):

        """
        Args:
            - pred: prediction probabilities (B, C, D, H, W)
            - target: one-hot encoded targets

        returns
            - tensor scalar of loss
        """

        dims = (2,3,4)

        TP = (pred*target).sum(dim=dims)
        FP = (pred * (1-target)).sum(dim=dims)
        FN = ((1-pred) * target).sum(dim=dims)

        alpha = torch.exp(self.log_alpha)
        beta = torch.exp(self.log_beta)

        tversky = (TP + self.eps) / (TP + alpha * FP + beta * FN + self.eps)

        loss = 1 - tversky.mean()

        reg = self.reg_weight * (alpha - beta).pow(2)

        return loss + reg


In [35]:
class CrossEntropyLoss(nn.Module):
    def __init__(self, reg_weight=0.0):
        """
        Args:
            - reg_weight: weight to prevent divergence

        """
        super().__init__()
        self.num_classes=4
        self.log_weights = nn.Parameter(torch.zeros(self.num_classes))
        self.reg_weight = reg_weight


    def forward(self, probs, target):
        """
        Args:
            - probs: predicted class probabilities
            - target: integer class labels
        """
        """
        weights = torch.exp(self.log_weights)
        weights = weights / weights.sum()      # normalize

        # per-voxel cross-entropy using negative log likelihood
        log_probs = torch.log(probs + 1e-8)  # for numerical stability
        loss = F.nll_loss(log_probs, target.long(), weight=weights, reduction='mean')

        # penalize imbalance between class weights
        reg = self.reg_weight * torch.var(weights) if self.reg_weight > 0 else 0.0

        """

        weights = torch.exp(self.log_weights)
        weights = weights / weights.sum()  # normalize to sum to 1

        # Compute NLL loss (cross entropy with log-probs)
        log_probs = torch.log(probs + 1e-8)  # avoid log(0)
        loss = F.nll_loss(log_probs, target.long(), weight=weights.detach(), reduction='mean')

        # Add regularization on the weights' variance to avoid collapse
        reg = self.reg_weight * torch.var(weights) if self.reg_weight > 0 else 0.0

        return loss + reg


In [36]:
class StructuralLoss(nn.Module):
    def __init__(self, dice_loss, ce_loss):
        super().__init__()

        self.dice_loss = dice_loss              # expects probs
        self.ce_loss = ce_loss                  # expects probs or logits,

        # Learnable weights (log-space for constraint: exp >= 0)
        self.alpha = nn.Parameter(torch.tensor(0.0))    # for Dice
        self.beta = nn.Parameter(torch.tensor(0.0))     # for CE


    def forward(self, preds, target):
        """
        Args:
            - preds: logits or probs depending on CE config
            - target: integer class labels (B, D, H, W)
        """
        probs = F.softmax(preds, dim=1)  # ensure softmax

        dice = self.dice_loss(probs, target)
        ce = self.ce_loss(probs, target)        # using prob-based CE

        total = self.alpha.exp() * dice + self.beta.exp() * ce


        logs = {
            'total_loss': total.item(),
            'dice_loss': dice.item(),
            'alpha_dice_loss': self.alpha.item(),
            'ce_loss': ce.item(),
            'beta_ce_loss': self.beta.item()
        }

        return total, logs


In [37]:
def trainv0(model, train_loader, loss_fn, optimizer, num_iterations, save_dir, device='cuda'):
    os.makedirs(save_dir, exist_ok=True)

    model.to(device)
    loss_fn.to(device)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=5, factor=0.5, verbose=True
    )

    best_loss = float('inf')
    no_improve = 0
    early_stop_patience = 10

    metric_logs = []

    for step, (inputs, targets) in enumerate(train_loader):
        if step >= num_iterations:
            break

        start_time = time.time()

        model.train()
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        preds = model(inputs)
        loss, loss_logs = loss_fn(preds, targets)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        scheduler.step(loss)

        step_time = time.time() - start_time
        current_lr = optimizer.param_groups[0]['lr']

        log_entry = {
            'step': step,
            'step_time_sec': round(step_time, 3),
            'loss': loss.item(),
            'lr': current_lr,
            **loss_logs
        }
        metric_logs.append(log_entry)

        # Save model
        checkpoint_path = os.path.join(save_dir, f'model_step{step}.pth')
        torch.save(model.state_dict(), checkpoint_path)

        # Save metrics
        with open(os.path.join(save_dir, 'metrics.json'), 'w') as f:
            json.dump(metric_logs, f, indent=2)

        # Logging to console
        ce = loss_logs.get('ce_loss', 0.0)
        dice = loss_logs.get('dice_loss', 0.0)
        #topo = loss_logs.get('topo_loss', 0.0)

        print(f"[{step:04d}] {step_time:.2f}s | "
              f"Loss: {loss.item():.4f} | Dice: {dice:.4f} | CE: {ce:.4f} |  | LR: {current_lr:.2e}")
    return metric_logs


In [38]:

model = UNet3D()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

tversky_dice = TverskyDiceLoss()
cross_entropy = CrossEntropyLoss()
#topo_loss = TopologicalLoss()
loss_fn = StructuralLoss(dice_loss=tversky_dice, ce_loss=cross_entropy)


In [39]:
trainv0(model, bt_dataloader, loss_fn, optimizer=optimizer, num_iterations=10,  save_dir='./checkpoint_path')

TypeError: ReduceLROnPlateau.__init__() got an unexpected keyword argument 'verbose'

`scikit-image` is an open-source Python library for image processing algorithms. It's built on NumPy, SciPy, and Matplotlib. It provides a wide range of functionalities including:

*   Image filtering
*   Image segmentation
*   Feature detection
*   Geometric transformations
*   Color space manipulation
*   And more.

To install `scikit-image`, you can use pip:

In [ ]:
from skimage import measure

In [ ]:
from kornia.morphology import dilation, erosion

In [ ]:
"""
Global Percentile-Based Normalization for Multimodal Brain MRI

This implementation addresses the fundamental challenge of preserving biological consistency
in multimodal medical image preprocessing. Unlike per-sample normalization approaches that
destroy the biological meaning of intensity combinations, this global normalization strategy
ensures that identical tissue signatures maintain consistent normalized representations
across all patients.

Core Principle: A voxel with intensities (100, 310, 1012, 421) across FLAIR, T1w, T1GD,
and T2w modalities represents a specific biological tissue signature. This signature should
normalize to the same values regardless of which patient scan it appears in, preserving
the quantitative biological information that enables robust medical image analysis.

The approach uses pre-computed global statistics across all 484 BRATS subjects to establish
consistent reference points for each modality, ensuring that biological interpretability
is maintained while achieving the statistical stability required for deep learning applications.
"""

import torch
from torch.utils.data import Dataset
import nibabel as nib
import os

# Pre-computed global statistics across all BRATS subjects
# These values represent the 0.5th and 99.5th percentiles for each modality,
# providing robust reference points that minimize outlier influence while
# preserving the full biological dynamic range of each imaging sequence
BRAINTUMOR_GLOBAL_STATS = {
    'FLAIR': {'p0.5': 23.958, 'p99.5': 1168.326},
    'T1w': {'p0.5': 45.15, 'p99.5': 2012.474},
    'T1GD': {'p0.5': 38.086, 'p99.5': 2305.63},
    'T2w': {'p0.5': 41.618, 'p99.5': 1669.87}
}

def percentile_normalization(data_tensor, modality):
    """
    Apply global percentile-based normalization to preserve biological consistency.

    This function implements the mathematical transform:
    v'_m = (clip(v_m, L_m, U_m) - L_m) / (U_m - L_m)

    where L_m and U_m are the global 0.5th and 99.5th percentiles for modality m,
    computed across the entire dataset. This ensures that identical raw intensity
    values always produce identical normalized representations, regardless of the
    patient sample they appear in.

    Args:
        data_tensor: 3D tensor containing MRI intensities for a single modality
        modality: String identifier ('FLAIR', 'T1w', 'T1GD', 'T2w')

    Returns:
        Normalized tensor with biological consistency preserved
    """
    # Extract the global percentile boundaries for this specific modality
    # These serve as the consistent reference frame across all patients
    params = BRAINTUMOR_GLOBAL_STATS[modality]
    p_low, p_high = params['p0.5'], params['p99.5']

    # Calculate the normalization range using global statistics
    # This range remains constant across all samples, ensuring biological consistency
    range_val = p_high - p_low

    # Create a mask to identify foreground voxels (tissue regions)
    # Background voxels (intensity = 0) carry no biological information and
    # would distort normalization statistics if included in calculations
    # This surgical precision ensures we only normalize biologically meaningful data
    mask = data_tensor > 0

    # Extract only the foreground (tissue) voxels for normalization
    # This prevents background regions from influencing the biological
    # interpretation of tissue intensity values
    foreground = data_tensor[mask]

    # Initialize the output tensor with zeros, preserving background regions
    # This maintains the spatial structure while allowing selective normalization
    # of tissue regions only
    normalized = torch.zeros_like(data_tensor)

    # Apply the global normalization transform to foreground voxels
    # Subtract the global lower bound and divide by the global range
    # This maps the robust percentile range [p_low, p_high] to [0, 1]
    scaled = (foreground - p_low) / range_val

    # Clamp values to [0, 1] range to handle any outliers beyond the percentile bounds
    # This provides robustness against extreme values while maintaining the
    # biological consistency principle - outliers get mapped to the boundary values
    # rather than distorting the entire normalization
    scaled = torch.clamp(scaled, 0, 1)

    # Place the normalized foreground values back into their original spatial positions
    # Background voxels remain zero, preserving the anatomical structure
    normalized[mask] = scaled

    return normalized


class BratsDataset(Dataset):
    """
    PyTorch Dataset class for BRATS multimodal brain MRI data with global normalization.

    This dataset implementation ensures biological consistency by applying the same
    global normalization parameters to each modality across all patients. The approach
    preserves the quantitative biological meaning of intensity combinations while
    providing the statistical stability required for deep learning applications.

    Key Design Principles:
    1. Biological Consistency: Identical tissue signatures normalize to identical values
    2. Modality-Specific Treatment: Each MRI sequence receives appropriate normalization
    3. Spatial Preservation: Anatomical relationships are maintained throughout processing
    """

    def __init__(self, images_dir, labels_dir):
        """
        Initialize the dataset with paths to image and label directories.

        The initialization process identifies common files between image and label
        directories to ensure proper pairing of multimodal data with ground truth
        segmentation masks.
        """
        super().__init__()

        self.images_dir = images_dir
        self.labels_dir = labels_dir

        # Identify all NIfTI files in both directories
        # NIfTI format is the standard for medical imaging data storage
        self.image_files = {f for f in os.listdir(self.images_dir) if f.endswith('.nii.gz')}
        self.label_files = {f for f in os.listdir(self.labels_dir) if f.endswith('.nii.gz')}

        # Find common files between images and labels to ensure proper pairing
        # This intersection operation guarantees that every sample has both
        # multimodal images and corresponding segmentation labels
        self.common_files = sorted(self.image_files & self.label_files)

        # Define the four MRI modalities in the standard BRATS order
        # Each modality captures different biological tissue properties:
        # - FLAIR: Fluid-attenuated inversion recovery (highlights abnormalities)
        # - T1w: T1-weighted (anatomical structure)
        # - T1GD: T1-weighted with gadolinium contrast (blood-brain barrier breakdown)
        # - T2w: T2-weighted (tissue water content, edema)
        self.modalities = ['FLAIR', 'T1w', 'T1GD', 'T2w']

    def __len__(self):
        """Return the total number of samples in the dataset."""
        return len(self.common_files)

    def __getitem__(self, index):
        """
        Load and preprocess a single sample with global normalization applied.

        This method implements the core data loading pipeline that preserves
        biological consistency while preparing data for deep learning models.
        The global normalization ensures that tissue signatures maintain their
        biological meaning across all patients in the dataset.
        """
        # Get the filename for the requested sample
        file_index = self.common_files[index]

        # Load the multimodal MRI image using nibabel
        # nibabel is the standard library for reading neuroimaging data formats
        image = nib.load(os.path.join(self.images_dir, file_index)).get_fdata()

        # Load the corresponding segmentation label
        # Labels contain ground truth tumor segmentation masks
        label = nib.load(os.path.join(self.labels_dir, file_index)).get_fdata()

        # Convert numpy arrays to PyTorch tensors and reorder dimensions
        # Original shape: (H, W, D, C) -> PyTorch convention: (C, D, W, H)
        # This permutation aligns with deep learning frameworks' expected input format
        # while preserving the spatial and modal relationships crucial for medical analysis
        image_tensor = torch.from_numpy(image).permute(3, 2, 0, 1).float()

        # Apply global normalization to each modality independently
        # This is the critical step that preserves biological consistency
        # Each modality is normalized using its global statistics, ensuring that
        # identical intensity values across patients receive identical normalized values
        for c, modality in enumerate(self.modalities):
            # Apply modality-specific global normalization to channel c
            # This ensures that the biological meaning of intensity combinations
            # remains consistent across all patients in the dataset
            image_tensor[c] = percentile_normalization(image_tensor[c], modality)

        # Prepare the label tensor with appropriate dimension ordering
        # Labels are single-channel, so we only need spatial dimension reordering
        label_tensor = torch.from_numpy(label)
        label_tensor = label_tensor.permute(2, 0, 1).long()

        # Return the globally normalized multimodal image and corresponding label
        # The image now maintains biological consistency while being statistically
        # prepared for robust deep learning model training
        return image_tensor, label_tensor